# 31 DPO evaluation (CPU runtime)

Scores the DPO model on validation, dev and mini-dev, and compares it on validation with the SFT model it started from, by the same rule as before: DPO is kept only if its validation EX is not significantly below the best and it has more answers at least 2x cheaper than gold. Dev and mini-dev are reported, never used to choose.

**Runtime > Change runtime type > CPU** (timing runs on the same machine type as all earlier timing).

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
POOL, EXPORT, DPO_EXPORT = "pool_v1", "sft_v1", "dpo_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# The SFT model and efficiency target kept on validation (28_sft_tag_choice): DPO starts from it.
CHOSEN = json.load(open(f"{R}/{EXPORT}_tag_choice/selection.json"))["winner"]
cm = json.load(open(f"{R}/{CHOSEN}/adapter_meta.json"))
INIT, TAG = cm.get("adapter_run", CHOSEN), cm.get("tag")
RUN = f"{DPO_EXPORT}_{cm['arm']}"
print("SFT model:", CHOSEN, "| adapter:", INIT, "| target:", TAG, "| DPO run:", RUN)
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
meta = json.load(open(f"{R}/{EXPORT}/export_meta.json"))
VAL_DBS = ",".join(meta["holdout_db_ids"])

### 1. Stage the validation databases and the dev databases

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev

### 2. Score the DPO model (resumable)

In [ ]:
!python -m effsql.train eval-val --run {RUN}
!effsql evaluate --pred {R}/{RUN}/predictions_dev.jsonl --split dev --timing none --run {RUN}_dev --model {MODEL} --method dpo-tagged-tag{TAG}
!effsql evaluate --pred {R}/{RUN}/predictions_minidev.jsonl --split minidev --timing quick --run {RUN}_minidev --model {MODEL} --method dpo-tagged-tag{TAG}

### 3. DPO against its SFT starting point on validation

In [ ]:
!python -m effsql.train select --runs {CHOSEN},{RUN} --out {DPO_EXPORT}_choice

### 4. Validation pair metrics during training, and all results

In [ ]:
import pandas as pd
print(pd.read_json(f"{R}/{RUN}/val_log.jsonl", lines=True).to_string(index=False))
df = pd.read_csv(f"{DRIVE_ROOT}/effsql/results/results.csv")
df = df[df["run"].str.startswith((RUN, CHOSEN, "pool_v1_"))].drop_duplicates("run", keep="last")
cols = ["run", "split", "n", "ex", "rel_cost_median", "rel_cost_geomean", "eff_acc@1.5", "quick_ratio_median"]
print(df[cols].to_string(index=False))

### Done: disconnect the runtime